In [1]:
# GPU only
import sys, os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress all TF messages except errors
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'  # Disable oneDNN messages
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Make sure GPU 0 is visible
os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=0'  # Disable XLA JIT to reduce warnings


In [2]:
# loading libraries for data manipulation
import numpy as np
import pandas as pd

# loading libraries for data visualization
import matplotlib.pyplot as plt
from plotnine import *
from PIL import Image

# import tensorflow and keras packages
import tensorflow as tf
from tensorflow import keras

# let's also include different Models, Layers directly from keras
from tensorflow.keras.models import Sequential,load_model
from tensorflow.keras.layers import Dense,Dropout,LSTM,Embedding,Input,GRU

# use requests package to download some text
import requests

import warnings
warnings.filterwarnings('ignore')

In [3]:
# Check GPU availability
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("\nGPU Details:")
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    print(f"  {gpu}")
    
# Optional: See detailed device placement during operations
# tf.debugging.set_log_device_placement(True)

tf.get_logger().setLevel('ERROR')
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)


Num GPUs Available:  1

GPU Details:
  PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


I0000 00:00:1763022414.899513     972 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1763022414.974844     972 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1763022414.974883     972 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


This notebook details the steps to train an LSTM to predict the next **word** given some input. We will use a larger corpus of text (Pride and Prejudice). 

In [4]:
# url to Pride and Prejudice in text form
url = "https://gutenberg.org/cache/epub/1342/pg1342.txt"
text = requests.get(url).text

# clean text 
text = text[text.find("Chapter I.]")+10:text.find("*** END OF THE PROJECT")] # exclude metadata
text = text.lower()
print(f"Length of text: {len(text)} characters")

Length of text: 708325 characters


In [5]:
# identify unique words in text
words = text.split()
print(f"Total words: {len(words)}")

Total words: 122410


In [6]:
# generate the two dictionaries
vocab = sorted(set(words))
print(f"Unique words: {len(vocab)}")

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}

Unique words: 12864


Now we can convert the entire text into a series of integers. Here, each word is represented by a unique integer ID. 

In [7]:
text_as_int = np.array([word2idx[w] for w in words], dtype=np.int32)
print("First 20 encoded words:", text_as_int[:20])

First 20 encoded words: [   34  6283  6274   218 11451 11657   345 11084   218 10253  6976  5856
  8627  7860   218  5018  4727  7504  1250  5856]


For this network, we will use a sequence length of 20 (words).

In [8]:
seq_length = 20  # smaller since words carry more info
examples_per_epoch = len(text_as_int) // (seq_length + 1)
print(f"Number of sequences: {examples_per_epoch}")

Number of sequences: 5829


Next, we will use tensorflow's from_tensor_slices function to create a stream of sequences. 

In [9]:
word_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = word_dataset.batch(seq_length + 1, drop_remainder=True)

I0000 00:00:1763022429.748663     972 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1763022429.748736     972 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1763022429.748751     972 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1763022429.892511     972 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1763022429.892566     972 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

In [10]:
# print the first words characters in the data
for i, item in enumerate(word_dataset.take(10)):
    print(item.numpy())

# print the first sequence 
for i, item in enumerate(sequences.take(1)):
    print(item.numpy())


34
6283
6274
218
11451
11657
345
11084
218
10253
[   34  6283  6274   218 11451 11657   345 11084   218 10253  6976  5856
  8627  7860   218  5018  4727  7504  1250  5856 11973]


Next, we can define a function that creates our dataset of sequences. 

In [11]:
#   input_text (first 20 chars)
#   target_text (the next 20 chars, shifted by one position)
def split_input_target(chunk):
    input_seq = chunk[:-1]
    target_seq = chunk[1:]
    return input_seq, target_seq

# apply the function to sequences
dataset = sequences.map(split_input_target)

In [12]:
for input_example, target_example in dataset.take(1):
    print("Input shape:", input_example.shape)
    print("Target shape:", target_example.shape)
    print("First input example (as IDs):", input_example[0].numpy())
    print("First target example (as IDs):", target_example[0].numpy())

Input shape: (20,)
Target shape: (20,)
First input example (as IDs): 34
First target example (as IDs): 6283


In [13]:
BATCH_SIZE = 64
BUFFER_SIZE = 10000
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

We have now created a dataset where each sequence is 20 words long and the target for that sequence is also 20 words long shifted by 1 word. We have also shuffled the input to the model to add some randomness. Note that buffer size if larger than the dataset size means an ideal situation for random selection. 

The Embedding layer will allow us to learn the relationship between characters. This is much better than one-hot encoding. So as part of predicting a sequence of characters, our model will also learn to better represent each character. 

In [14]:
# define hyperparameters for the network
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 512

model = Sequential([
    Input(shape=(None,)),
    Embedding(vocab_size, embedding_dim),
    LSTM(rnn_units, return_sequences=True),
    Dropout(0.2),
    Dense(vocab_size)
])

model.compile(
    optimizer='adam',
    loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True)
)


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


In [16]:
# Force TensorFlow to use standard GPU LSTM instead of cuDNN
# by violating one of the cuDNN optimization conditions

model = Sequential([
    Input(shape=(None,)),
    Embedding(vocab_size, embedding_dim),
    LSTM(rnn_units, return_sequences=True, recurrent_dropout=0.1),  # This breaks cuDNN trigger!
    Dropout(0.2),
    Dense(vocab_size)
])

model.compile(
    optimizer='adam',
    loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True)
)

In [17]:
# train model
history = model.fit(dataset, epochs=20,verbose=1)

Epoch 1/20
91/91 [==============================] - 9s 73ms/step - loss: 7.5096
Epoch 2/20
91/91 [==============================] - 7s 70ms/step - loss: 6.7980
Epoch 3/20
91/91 [==============================] - 5s 58ms/step - loss: 6.5642
Epoch 4/20
91/91 [==============================] - 6s 69ms/step - loss: 6.3555
Epoch 5/20
91/91 [==============================] - 3s 32ms/step - loss: 6.1565
Epoch 6/20
91/91 [==============================] - 6s 66ms/step - loss: 5.9212
Epoch 7/20
91/91 [==============================] - 6s 68ms/step - loss: 5.7196
Epoch 8/20
91/91 [==============================] - 6s 60ms/step - loss: 5.5477
Epoch 9/20
91/91 [==============================] - 6s 63ms/step - loss: 5.3921
Epoch 10/20
91/91 [==============================] - 3s 31ms/step - loss: 5.2528
Epoch 11/20
91/91 [==============================] - 5s 53ms/step - loss: 5.1198
Epoch 12/20
91/91 [==============================] - 6s 68ms/step - loss: 4.9963
Epoch 13/20
91/91 [==================

In [18]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, None, 256)         3293184   
                                                                 
 lstm_1 (LSTM)               (None, None, 512)         1574912   
                                                                 
 dropout_1 (Dropout)         (None, None, 512)         0         
                                                                 
 dense_1 (Dense)             (None, None, 12864)       6599232   
                                                                 
Total params: 11467328 (43.74 MB)
Trainable params: 11467328 (43.74 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [15]:
# if training takes too long, load pretrained model instead
model = load_model("pride_lstm_word_model.keras")

ValueError: File not found: filepath=pride_lstm_word_model.keras. Please ensure the file is an accessible `.keras` zip file.

In [16]:
def generate_text(model, start_seq, num_generate=50, temperature=1.0):
    # Tokenize the starting sequence into words
    input_eval = [word2idx.get(w, 0) for w in start_seq.lower().split()]
    input_eval = tf.expand_dims(input_eval, 0)

    generated_words = []

    for _ in range(num_generate):
        predictions = model.predict(input_eval, verbose=0)
        predictions = tf.squeeze(predictions, 0)
        predictions = predictions / temperature

        predicted_id = tf.random.categorical(predictions[-1:], num_samples=1)[0, 0].numpy()

        input_eval = tf.expand_dims([predicted_id], 0)
        generated_words.append(idx2word[predicted_id])

    return start_seq + ' ' + ' '.join(generated_words)

In [17]:
generate_text(model, "jane remained", 10, 1.0)

'jane remained surprised i am now then lucases. friendly “who convinced reply,'

In [19]:
generate_text(model,"he was",5,5.0)

'he was jilt it? deciding junior, forget'

In [18]:
generate_text(model,"it was",5,0.1)

'it was not be in the whole'

In [20]:
output = generate_text(model, "jane ", num_generate=1000, temperature=0.5)
output = output.split(".")
for sentence in output:
    print(sentence)

jane  had been in the room
 “i am not to be a persuasion that the principal abode
 that he was in the man of her husband’s
 you will be the winter, stone forbid that she was in the period of the day to be to introduce her the whole of the little of her mother was not be in the day to be as to bear in the coach; proceedings, wisely at the wife, self-command, to be a few minutes of the day to his friend: unimportant, confess, of the table, building, her brother was not talk of a disgrace; horseback, whisper,-- round” knowing what she was in the case, i will be in the whole of the end of her surprise in the first very abhorrent of, “must spring apartments likely to the whole of the little of the stream of the whole of the whole of the dinner gave her woman which you was to say of her to be a man in the profession which she was not the rest, supplying of my character of the want of the room
 elizabeth could not be so much to the easiness, sister?” dislike
 the whole of the day as to be eno